# WordPiece Tokenizer — From Scratch

## 1. Training corpus

In [3]:
corpus = [
    "low lower lowest",
    "playing played player",
    "walk walking walked",
    "talk talking talked",
    "unhappy happiness happy",
]

corpus

['low lower lowest',
 'playing played player',
 'walk walking walked',
 'talk talking talked',
 'unhappy happiness happy']

## 2. Pre-tokenize and initialize character-level pieces

In [4]:
import re

def preprocess_text(text):
    text = re.sub(r"[^\w\s]", " ", text)
    return text.lower()

def initialize_word(word):
    return [word[0]] + [f"##{char}" for char in word[1:]]

def initialize_corpus(corpus):
    words = []
    for sentence in corpus:
        words.extend(preprocess_text(sentence).split())
    return words

words = initialize_corpus(corpus)
word_splits = {word: initialize_word(word) for word in words}

word_splits

{'low': ['l', '##o', '##w'],
 'lower': ['l', '##o', '##w', '##e', '##r'],
 'lowest': ['l', '##o', '##w', '##e', '##s', '##t'],
 'playing': ['p', '##l', '##a', '##y', '##i', '##n', '##g'],
 'played': ['p', '##l', '##a', '##y', '##e', '##d'],
 'player': ['p', '##l', '##a', '##y', '##e', '##r'],
 'walk': ['w', '##a', '##l', '##k'],
 'walking': ['w', '##a', '##l', '##k', '##i', '##n', '##g'],
 'walked': ['w', '##a', '##l', '##k', '##e', '##d'],
 'talk': ['t', '##a', '##l', '##k'],
 'talking': ['t', '##a', '##l', '##k', '##i', '##n', '##g'],
 'talked': ['t', '##a', '##l', '##k', '##e', '##d'],
 'unhappy': ['u', '##n', '##h', '##a', '##p', '##p', '##y'],
 'happiness': ['h', '##a', '##p', '##p', '##i', '##n', '##e', '##s', '##s'],
 'happy': ['h', '##a', '##p', '##p', '##y']}

## 3. Count subwords and adjacent pairs

In [5]:
from collections import Counter

word_freq = Counter(words)

def get_subword_counts(splits, word_freq):
    counts = Counter()
    for word, pieces in splits.items():
        for piece in pieces:
            counts[piece] += word_freq[word]
    return counts

def get_pair_counts(splits, word_freq):
    counts = Counter()
    for word, pieces in splits.items():
        for i in range(len(pieces) - 1):
            pair = (pieces[i], pieces[i + 1])
            counts[pair] += word_freq[word]
    return counts

subword_counts = get_subword_counts(word_splits, word_freq)
pair_counts = get_pair_counts(word_splits, word_freq)

print("Subword counts:", subword_counts)
print("\nPair counts:", pair_counts)

Subword counts: Counter({'##a': 12, '##l': 9, '##e': 7, '##k': 6, '##p': 6, '##y': 5, '##n': 5, '##i': 4, 'l': 3, '##o': 3, '##w': 3, '##s': 3, 'p': 3, '##g': 3, '##d': 3, 'w': 3, 't': 3, '##r': 2, 'h': 2, '##t': 1, 'u': 1, '##h': 1})

Pair counts: Counter({('##a', '##l'): 6, ('##l', '##k'): 6, ('##i', '##n'): 4, ('l', '##o'): 3, ('##o', '##w'): 3, ('p', '##l'): 3, ('##l', '##a'): 3, ('##a', '##y'): 3, ('##n', '##g'): 3, ('##e', '##d'): 3, ('w', '##a'): 3, ('t', '##a'): 3, ('##a', '##p'): 3, ('##p', '##p'): 3, ('##w', '##e'): 2, ('##e', '##r'): 2, ('##e', '##s'): 2, ('##y', '##e'): 2, ('##k', '##i'): 2, ('##k', '##e'): 2, ('##p', '##y'): 2, ('h', '##a'): 2, ('##s', '##t'): 1, ('##y', '##i'): 1, ('u', '##n'): 1, ('##n', '##h'): 1, ('##h', '##a'): 1, ('##p', '##i'): 1, ('##n', '##e'): 1, ('##s', '##s'): 1})


## 4. WordPiece merge score

In [ ]:
def score_pairs(pair_counts, subword_counts):
    scores = {}
    for (left, right), pair_count in pair_counts.items():
        scores[(left, right)] = pair_count / (
            subword_counts[left] * subword_counts[right]
        )
    return scores

def merge_pair_in_word(pieces, pair_to_merge):
    merged_pieces = []
    i = 0

    while i < len(pieces):
        if i < len(pieces) - 1 and (pieces[i], pieces[i + 1]) == pair_to_merge:
            merged = pieces[i] + pieces[i + 1].replace("##", "", 1)
            merged_pieces.append(merged)
            i += 2
        else:
            merged_pieces.append(pieces[i])
            i += 1

    return merged_pieces

def merge_pair_in_corpus(splits, pair_to_merge):
    new_splits = {}
    for word, pieces in splits.items():
        new_splits[word] = merge_pair_in_word(pieces, pair_to_merge)
    return new_splits

def train_wordpiece(corpus, target_vocab_size=50, verbose=True):
    words = initialize_corpus(corpus)
    word_freq = Counter(words)

    splits = {word: initialize_word(word) for word in word_freq}
    vocab = set(piece for pieces in splits.values() for piece in pieces)

    while len(vocab) < target_vocab_size:
        subword_counts = get_subword_counts(splits, word_freq)
        pair_counts = get_pair_counts(splits, word_freq)

        if not pair_counts:
            break

        scores = score_pairs(pair_counts, subword_counts)
        best_pair = max(scores, key=scores.get)

        if verbose:
            print(f"Merge {best_pair} | score={scores[best_pair]:.6f}")

        splits = merge_pair_in_corpus(splits, best_pair)

        merged_token = best_pair[0] + best_pair[1].replace("##", "", 1)
        vocab.add(merged_token)

    return vocab, splits

vocab, trained_splits = train_wordpiece(corpus, target_vocab_size=45)

print("\nVocabulary size:", len(vocab))
print("Vocabulary:")
print(sorted(vocab))

## 5. Tokenize a word (greedy longest-match-first)

In [ ]:
import re

def preprocess_text(text):
    text = re.sub(r"[^\w\s]", " ", text)
    return text.lower()


def tokenize_word(word, vocab):
    word = preprocess_text(word).strip()

    if not word:
        return []

    tokens = []
    start = 0

    while start < len(word):
        end = len(word)
        found = None

        while end > start:
            candidate = word[start:end]

            if start > 0:
                candidate = "##" + candidate

            if candidate in vocab:
                found = candidate
                break

            end -= 1

        if found is None:
            return ["[UNK]"]

        tokens.append(found)
        start = end

    return tokens


def tokenize_text(text, vocab):
    cleaned_text = preprocess_text(text)

    return [
        token
        for word in cleaned_text.split()
        for token in tokenize_word(word, vocab)
    ]

## 6. Explore vocabulary size vs. tokenization

In [ ]:
small_vocab, _ = train_wordpiece(corpus, target_vocab_size=25, verbose=False)
larger_vocab, _ = train_wordpiece(corpus, target_vocab_size=45, verbose=False)

test_words = ["playing", "played", "walking", "unhappy", "happiness"]

print("Smaller vocabulary:")
for word in test_words:
    print(word, "->", tokenize_word(word, small_vocab))

print("\nLarger vocabulary:")
for word in test_words:
    print(word, "->", tokenize_word(word, larger_vocab))